<a href="https://colab.research.google.com/github/shivansh2310/The-elements-of-quantitative-investing-/blob/main/Sizing%2C_Allocation_%26_Ex_Post_Attribution_(Chapters_13%E2%80%9314).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Sizing with the Kelly Criterion

Paleologo emphasizes that having a positive expected return is useless if your bet sizing guarantees a margin call during a routine drawdown.

* The Kelly Criterion: The mathematical formula that maximizes the long-term compound growth rate of your bankroll. In continuous finance, the optimal fraction of your capital to allocate to a strategy is $f^* = \mu / \sigma^2$ (Expected Excess Return divided by Variance).

* The Reality (Fractional Kelly): Full Kelly is notoriously aggressive. Because we are only estimating $\mu$ and $\sigma$ (and we are often wrong), betting "Full Kelly" results in terrifying volatility and near-100% drawdowns. Institutional quants almost universally trade Fractional Kelly (e.g., Half-Kelly or Quarter-Kelly) to trade a slightly lower compound growth rate for a massive reduction in drawdowns.

### Ex-Post Performance Attribution

If your FMP makes 2% next month, you need to prove to your investors why you made that 2%. Did the "Value" factor actually work, or did you just get lucky because JPM crushed its earnings report?

Total PnL = Factor PnL + Specific (Idiosyncratic) PnL

* Factor PnL: The money you made purely from your exposure to the structural factors (Market and Value).

* Specific PnL: The money you made (or lost) because the individual stocks deviated from the structural model.

If your FMP is built correctly, the Specific PnL should roughly net out to zero over time, and your returns should be driven entirely by the Factor PnL.

### PnL Attribution

In [20]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from sklearn.covariance import LedoitWolf
import warnings

warnings.filterwarnings('ignore') # Clean output

In [21]:
# ==========================================
stock_data = {
    "MSFT": "Tech", "NVDA": "Tech", "AAPL": "Tech",
    "NEE": "Utilities", "AEP": "Utilities", "VST": "Utilities",
    "CAT": "Industrials", "GE": "Industrials", "PWR": "Industrials",
    "JPM": "Financials", "GS": "Financials", "V": "Financials",
    "XOM": "Energy", "CVX": "Energy",
    "META": "Comm", "GOOGL": "Comm",
    "LLY": "Healthcare", "UNH": "Healthcare",
    "AMZN": "Consumer", "WMT": "Consumer"
}
tickers = list(stock_data.keys())

print("Fetching Market Data & Splitting Train/Test... (Takes ~15 secs)")
# Fetch 2 years of data
all_prices = yf.download(tickers, period="2y")['Close']
all_prices = all_prices[tickers] # Force strict column alignment

# Train/Test Split (Preventing Look-Ahead Bias)
# Train: Everything up to the last 21 trading days (~1 month)
# Test: The last 21 trading days (Out of sample)
train_prices = all_prices.iloc[:-21]
test_prices = all_prices.iloc[-21:]

train_returns = train_prices.pct_change().dropna()
test_returns = test_prices.pct_change().dropna()

[                       0%                       ]

Fetching Market Data & Splitting Train/Test... (Takes ~15 secs)


[*********************100%***********************]  20 of 20 completed


In [22]:
print("Fetching Fundamentals & Building Exposures...")
fund_data = []
for ticker in tickers:
    try:
        tk = yf.Ticker(ticker)
        pe = tk.info.get('trailingPE', np.nan)
        ey = 1 / pe if pd.notna(pe) and pe > 0 else np.nan
        fund_data.append({"Ticker": ticker, "EarningsYield": ey})
    except Exception:
        fund_data.append({"Ticker": ticker, "EarningsYield": np.nan})

df = pd.DataFrame(fund_data).set_index("Ticker")
df['EarningsYield'] = df['EarningsYield'].fillna(df['EarningsYield'].mean())

# Z-Score and Build Exposure Matrix (X)
df['Value_Z'] = (df['EarningsYield'] - df['EarningsYield'].mean()) / df['EarningsYield'].std()
X_fmp = sm.add_constant(df[['Value_Z']])
X_matrix = X_fmp.values
X_matrix_df = pd.DataFrame(X_matrix, index=tickers, columns=['Beta', 'Value_Z'])

Fetching Fundamentals & Building Exposures...


In [23]:
print("Calculating Covariance & Engineering FMP...")
# Fit Ledoit-Wolf strictly on the TRAIN returns
shrunk_cov_daily = LedoitWolf().fit(train_returns).covariance_
cov_annual = shrunk_cov_daily * 252

Sigma = cov_annual
Sigma_inv = np.linalg.inv(Sigma)

# The Core FMP Equation
XT_SigmaInv_X = X_matrix.T @ Sigma_inv @ X_matrix
XT_SigmaInv_X_inv = np.linalg.inv(XT_SigmaInv_X)
H_matrix = Sigma_inv @ X_matrix @ XT_SigmaInv_X_inv

# Extract Pure Value FMP Weights (Index 1)
fmp_weights = H_matrix[:, 1]
fmp_df = pd.Series(fmp_weights, index=tickers)

Calculating Covariance & Engineering FMP...


In [24]:
print("Running Out-of-Sample Ex-Post Attribution...\n")

# Calculate cumulative returns for the TEST period (last 1 month)
cum_test_returns = (1 + test_returns).prod() - 1

# Total Strategy PnL
total_pnl = (fmp_df * cum_test_returns).sum()

# Extract Realized Factor Returns (Cross-sectional regression on the TEST period)
realized_model = sm.OLS(cum_test_returns, X_matrix_df).fit()
realized_factor_returns = realized_model.params

# Factor PnL (Returns generated by our structural exposure)
portfolio_exposures = fmp_df.T @ X_matrix_df
factor_pnl = (portfolio_exposures * realized_factor_returns).sum()

# Specific PnL (Returns generated by idiosyncratic noise)
specific_pnl = total_pnl - factor_pnl

Running Out-of-Sample Ex-Post Attribution...



In [25]:
# ==========================================
print("="*55)
print(" FINAL QUANTS REPORT: OUT-OF-SAMPLE ATTRIBUTION")
print("="*55)
print("PORTFOLIO STRUCTURE:")
print(f"Net Market Exposure:    {fmp_df.sum():.6f} (Should be ~0)")
print(f"Gross Leverage:         {fmp_df.abs().sum():.4f}")
print("-" * 55)
print(f"Total Strategy Return:  {total_pnl * 100:>8.4f}%")
print(f"Factor Contribution:    {factor_pnl * 100:>8.4f}%")
print(f"Specific Contribution:  {specific_pnl * 100:>8.4f}%")
print("="*55)

if abs(factor_pnl) > abs(specific_pnl):
    print("\nCONCLUSION: STRUCTURAL SUCCESS")
    print("Your PnL was driven primarily by your engineered factor exposure.")
    print("The FMP successfully isolated the alpha.")
else:
    print("\nCONCLUSION: HIGH NOISE ENVIRONMENT")
    print("Your PnL was heavily influenced by idiosyncratic stock movements ")
    print("rather than the intended factor. Consider increasing the universe size")
    print("to diversify away specific risk.")

 FINAL QUANTS REPORT: OUT-OF-SAMPLE ATTRIBUTION
PORTFOLIO STRUCTURE:
Net Market Exposure:    0.000000 (Should be ~0)
Gross Leverage:         0.8870
-------------------------------------------------------
Total Strategy Return:    0.3683%
Factor Contribution:      0.7381%
Specific Contribution:   -0.3698%

CONCLUSION: STRUCTURAL SUCCESS
Your PnL was driven primarily by your engineered factor exposure.
The FMP successfully isolated the alpha.
